# Chapter 4B Track 2: GRPO Fine-Tuning for Schema Linking

**Goal**: Train a model to predict which tables and columns are needed to answer a SQL question, using reinforcement learning (GRPO) instead of supervised learning (SFT).

```
Input:  Question + Database Schema

Output: Tables: invoice, customer
        Columns: invoice.total, invoice.customer_id, customer.customer_id, customer.first_name
```

**How this differs from Track 1 (SFT)**:
- Track 1 imitates labeled `(question, SQL)` pairs — the model copies the training data
- Track 2 uses a **reward function** that scores predictions (F2 on tables + columns) — the model learns what works
- GRPO generates multiple candidate outputs per question, scores them, and reinforces the best ones

| | Track 1: SFT | Track 2: GRPO |
|---|---|---|
| Task | Text-to-SQL generation | Schema linking (table/column prediction) |
| Model | Qwen2.5-Coder-1.5B | Qwen3.5-2B |
| Data | (question, SQL) pairs | question + reward signal |
| Training | Supervised (cross-entropy) | RL (group-relative advantage) |
| Framework | Unsloth + SFTTrainer | Unsloth + GRPOTrainer |

- **Model**: `unsloth/Qwen3.5-2B` (LoRA fine-tuning)
- **Dataset**: `data/schema_linking_train.jsonl` (813 examples from the Chinook database)
- **Reward**: F2 score on predicted vs gold tables and columns
- **Hardware**: Colab A100 or local GPU with 24+ GB VRAM
- **Standalone**: No dependency on the book's `src/` modules

## 1. Install Dependencies

In [ ]:
%%capture
import os, importlib.util

# Qwen3.5 requires transformers>=5.2.0 + flash-linear-attention for DeltaNet layers.
# Unsloth's PyPI release pins transformers<=4.57.6, so we install from git.
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps tokenizers trl==0.24.0 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is needed for Qwen3.5's hybrid DeltaNet layers (torch==2.8.0 only)
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!uv pip install mergekit datasets wandb

# Patch trl: transformers>=5.x changed _is_package_available() to return (bool, version)
# instead of bool. trl's guards break because non-empty tuples are truthy.
import importlib, pathlib, re
_trl_path = pathlib.Path(importlib.util.find_spec("trl").origin).parent / "import_utils.py"
_src = _trl_path.read_text()
if "_raw_is_package_available" not in _src:
    _src = _src.replace(
        "from transformers.utils.import_utils import _is_package_available",
        "from transformers.utils.import_utils import _is_package_available as _raw_is_package_available\n"
        "def _is_package_available(name, return_version=False):\n"
        "    result = _raw_is_package_available(name)\n"
        "    if isinstance(result, tuple):\n"
        "        available, version = result\n"
        "    else:\n"
        "        available, version = result, None\n"
        "    if return_version:\n"
        "        return available, version\n"
        "    return available",
        1,
    )
    _trl_path.write_text(_src)
    print("Patched trl import_utils for transformers>=5.x compatibility")

## 2. Configuration

In [ ]:
# ============= CONFIGURATION =============

MODEL_NAME = "unsloth/Qwen3.5-2B"

# Data paths — relative to repo root (scripts/chapter_4B/)
TRAIN_DATA = "../../data/schema_linking_train.jsonl"
VAL_DATA = "../../data/schema_linking_val.jsonl"

# Sequence length — the full Chinook DDL schema is ~4,500 tokens per prompt.
# With 512 completion tokens, total is ~5,000. Use 5120 for headroom.
MAX_SEQ_LENGTH = 5120

# LoRA — same rank/alpha as Track 1 SFT notebook
LORA_R = 16
LORA_ALPHA = 16

# GRPO hyperparameters
NUM_GENERATIONS = 4       # Candidates per prompt (group size G)
MAX_COMPLETION_LENGTH = 512  # Must exceed actual generation length.
                             # Qwen3.5 wraps output in <think>...</think>,
                             # so actual tokens = response + thinking overhead.
NUM_EPOCHS = 2            # 2 epochs; 3rd often degrades (see 4C.3)
BATCH_SIZE = 2            # Prompts per batch (kept small — each prompt
                          # spawns NUM_GENERATIONS completions at 5120 tokens)
GRAD_ACCUM_STEPS = 4      # Effective batch = 2 * 4 = 8 prompts
LEARNING_RATE = 5e-6      # Lower than SFT — RL gradients are noisier
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

EVAL_EXAMPLES = 126
OUTPUT_DIR = "output_grpo_schema_linking"

# Weights & Biases — set to True to log metrics to wandb
USE_WANDB = True
WANDB_PROJECT = "grpo-schema-linking"

# Prompt template (inlined from scripts/chapter_4C/schema_linking_prompt.txt)
SYSTEM_PROMPT = """You are a schema linking agent.

Given a question and database schema, list which tables and columns would be needed to answer the question.

Include ALL columns used anywhere: SELECT, JOIN, WHERE, GROUP BY, ORDER BY.
For columns inside functions like SUM(invoice.total), list only the column: invoice.total
Do NOT include function names, aliases, or SQL expressions.

CRITICAL: Every column MUST be written as table_name.column_name (e.g., invoice.total, customer.first_name). Never write a bare column name without its table prefix.

Do NOT write SQL. Only list the tables and columns.

Reply in exactly this format:
Tables: <comma-separated table names>
Columns: <comma-separated table.column names>

Example:
Tables: invoice, customer
Columns: invoice.total, invoice.billing_country, customer.customer_id, customer.first_name"""

## 2b. Weights & Biases Setup

wandb tracks reward curves, learning rate schedules, and GPU utilization across training runs. Set `USE_WANDB = False` in the config cell above to disable.

In [ ]:
import wandb
import os

if USE_WANDB:
    wandb.login()  # Uses WANDB_API_KEY env var, or prompts for key
    wandb.init(
        project=WANDB_PROJECT,
        config={
            "model": MODEL_NAME,
            "lora_r": LORA_R,
            "lora_alpha": LORA_ALPHA,
            "num_generations": NUM_GENERATIONS,
            "max_completion_length": MAX_COMPLETION_LENGTH,
            "epochs": NUM_EPOCHS,
            "batch_size": BATCH_SIZE,
            "grad_accum_steps": GRAD_ACCUM_STEPS,
            "effective_batch": BATCH_SIZE * GRAD_ACCUM_STEPS,
            "learning_rate": LEARNING_RATE,
            "warmup_ratio": WARMUP_RATIO,
            "weight_decay": WEIGHT_DECAY,
            "reward": "F2 (0.4*table + 0.6*column) + 0.1*format",
        },
    )
    print(f"wandb run: {wandb.run.url}")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("wandb disabled — set USE_WANDB = True to enable")

## 3. GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected.")
    print("On Colab: Runtime > Change runtime type > A100 GPU")

## 4. Load Model + Attach LoRA Adapters

Qwen3.5-2B is registered as a vision-language model (`Qwen3_5ForConditionalGeneration`) in HuggingFace — even for text-only use. Unsloth routes it through `FastVisionModel`, not `FastLanguageModel`. Two things to know:

1. **BF16 LoRA, not QLoRA**: Unsloth docs recommend against 4-bit quantisation (QLoRA) for Qwen3.5 — use `load_in_4bit=False` for 16-bit LoRA.
2. **Text-only LoRA**: Since schema linking is a text task, we set `finetune_vision_layers=False` and `finetune_language_layers=True` to only train the language model parts.

VERL also requires standard transformer architectures for its 3D-HybridEngine parallelism, which is why Qwen3.5-2B cannot be trained through the VERL pipeline (Chapter 4C.3). Unsloth's GRPOTrainer runs on a single GPU, sidestepping both limitations.

In [ ]:
from unsloth import FastVisionModel
import torch

model, processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit=False,                    # 16-bit LoRA (QLoRA not recommended for Qwen3.5)
    use_gradient_checkpointing="unsloth",  # Reduces VRAM, extends context
)

# FastVisionModel returns a processor, not a tokenizer.
# Extract the tokenizer for text-only usage (chat templates, encode/decode).
tokenizer = processor.tokenizer

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,      # Text-only task — skip vision layers
    finetune_language_layers=True,     # Train language model
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

### Fix: Reset `rope_deltas` for text-only VLM training

Qwen3.5 is a VLM — its `compute_3d_position_ids` caches `rope_deltas` during `model.generate()` for 3D vision position encoding. When GRPOTrainer then calls the model forward for logprob computation (with a different batch size and no images), the stale `rope_deltas` causes a tensor shape mismatch.

**Fix**: register a forward pre-hook that resets `rope_deltas = None` before each forward pass. This forces the model to skip 3D position encoding and use standard 1D positions for text-only data. Safe because during `generate()`, position_ids are always passed explicitly.

In [ ]:
# Qwen3.5 is a VLM that caches rope_deltas for 3D vision position encoding.
# During text-only GRPO, model.generate() sets rope_deltas, then the trainer's
# logprob computation calls forward() with a different batch size — causing a
# tensor shape mismatch in compute_3d_position_ids.
#
# Fix: reset rope_deltas before each forward so the model uses standard 1D positions.

# Walk through model wrappers to find the Qwen3_5Model (has rope_deltas)
inner_model = model
for attr in ["base_model", "model", "model"]:
    if hasattr(inner_model, attr):
        inner_model = getattr(inner_model, attr)
    if hasattr(inner_model, "rope_deltas"):
        break

assert hasattr(inner_model, "rope_deltas"), (
    f"Could not find rope_deltas on {type(inner_model).__name__}. "
    "Model wrapping may have changed — check model.base_model.model.model"
)

def _reset_rope_deltas(module, args, kwargs):
    """Pre-forward hook: clear stale rope_deltas cache for text-only inputs."""
    module.rope_deltas = None

inner_model.register_forward_pre_hook(_reset_rope_deltas, with_kwargs=True)
print(f"Registered rope_deltas reset hook on {type(inner_model).__name__}")

## 5. Load Data

Each example has:
- `question`: natural language question
- `ddl_context`: database schema (DDL with column descriptions)
- `glossary_context`: business term definitions
- `gold_tables`: list of correct table names
- `gold_columns`: list of correct `table.column` names

GRPOTrainer needs a `prompt` column (the formatted chat messages) plus any extra columns the reward function needs (`gold_tables`, `gold_columns`).

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    """Load a JSONL file into a list of dicts."""
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

def format_prompt(example):
    """Build chat messages for schema linking.

    GRPOTrainer expects 'prompt' to be a list of message dicts,
    NOT a pre-formatted string. trl applies the chat template internally.
    """
    user_content = (
        f"<SCHEMA>\n{example['ddl_context']}\n</SCHEMA>\n\n"
        f"<GLOSSARY>\n{example['glossary_context']}\n</GLOSSARY>\n\n"
        f"Question: {example['question']}"
    )
    example["prompt"] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    return example

# Load and format
train_records = load_jsonl(TRAIN_DATA)
val_records = load_jsonl(VAL_DATA)

train_ds = Dataset.from_list(train_records).map(format_prompt)
val_ds = Dataset.from_list(val_records).map(format_prompt)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)}")
print(f"\nSample question: {train_ds[0]['question']}")
print(f"Gold tables: {train_ds[0]['gold_tables']}")
print(f"Gold columns: {train_ds[0]['gold_columns'][:5]}...")
print(f"\nPrompt format: list of {len(train_ds[0]['prompt'])} message dicts")
print(f"  roles: {[m['role'] for m in train_ds[0]['prompt']]}")

## 6. Reward Function

The reward function scores model predictions against gold tables and columns using F2 score plus a format bonus:

```
content_score = 0.4 * table_F2 + 0.6 * column_F2
total = 0.9 * content_score + 0.1 * format_score
```

**Why F2?** In schema linking, missing a table/column is worse than including an extra one. F2 (beta=2) weights recall 2x over precision, matching this asymmetry.

**Why format score?** In early training, the model may not know the expected output format. The 0.1 format weight provides gradient signal even when content is wrong, preventing the reward from being 0 for every output.

This is the same reward function used in Chapter 4C.3 (`scripts/chapter_4C/reward_schema_linking.py`), adapted for GRPOTrainer's `(completions, **kwargs) -> list[float]` signature.

In [ ]:
import re

# ---------- Parsing ----------

def _add_items(text, target):
    """Split comma-separated items and add non-empty ones to the target set."""
    for item in text.split(","):
        item = item.strip()
        if item:
            target.add(item)


def parse_prediction(text):
    """Parse model output into predicted tables and columns.

    Handles comma-separated, bullet, and numbered formats.
    Returns (pred_tables, pred_columns) as sets.
    """
    tables = set()
    columns = set()
    lines = text.strip().splitlines()
    current_section = None

    for line in lines:
        line = line.strip()
        if not line:
            continue

        tables_match = re.match(r"^tables?\s*:\s*(.*)$", line, re.IGNORECASE)
        cols_match = re.match(r"^columns?\s*:\s*(.*)$", line, re.IGNORECASE)

        if tables_match:
            current_section = "tables"
            rest = tables_match.group(1).strip()
            if rest:
                _add_items(rest, tables)
            continue

        if cols_match:
            current_section = "columns"
            rest = cols_match.group(1).strip()
            if rest:
                _add_items(rest, columns)
            continue

        if current_section and re.match(r"^[-*]\s+|^\d+[.)]\s+", line):
            item = re.sub(r"^[-*]\s+|^\d+[.)]\s+", "", line).strip()
            if item:
                target = tables if current_section == "tables" else columns
                _add_items(item, target)
        else:
            current_section = None

    return tables, columns


# ---------- F-beta score ----------

BETA = 2.0

def _compute_f_beta(pred_set, gold_set, beta=BETA):
    """Compute F-beta score. beta > 1 weights recall more than precision."""
    if not pred_set and not gold_set:
        return 1.0
    if not pred_set or not gold_set:
        return 0.0
    overlap = len(pred_set & gold_set)
    precision = overlap / len(pred_set)
    recall = overlap / len(gold_set)
    if precision + recall == 0:
        return 0.0
    b2 = beta * beta
    return (1 + b2) * precision * recall / (b2 * precision + recall)


def _strip_function_wrapper(col):
    """Strip SQL function wrappers: sum(invoice.total) -> invoice.total."""
    m = re.match(r"^[a-z_]+\((.+)\)$", col.strip(), re.IGNORECASE)
    if m:
        return m.group(1).strip()
    return col


def _normalize_columns(cols):
    """Normalize columns: lowercase, strip wrappers."""
    return {_strip_function_wrapper(c.strip()).lower().strip() for c in cols if c.strip()}


def _normalize_tables(tables):
    """Normalize tables: lowercase, strip whitespace."""
    return {t.lower().strip() for t in tables if t.strip()}


def _compute_format_score(text):
    """Score format adherence (0.0-1.0)."""
    has_tables = bool(re.search(r"^tables?\s*:", text, re.IGNORECASE | re.MULTILINE))
    has_columns = bool(re.search(r"^columns?\s*:", text, re.IGNORECASE | re.MULTILINE))
    if has_tables and has_columns:
        return 1.0
    if has_tables or has_columns:
        return 0.5
    return 0.0


# ---------- Single-example reward ----------

def compute_schema_linking_reward(predicted_text, gold_tables, gold_columns):
    """Compute schema linking reward for one prediction.

    Returns float: 0.9 * content_score + 0.1 * format_score
    where content_score = 0.4 * table_F2 + 0.6 * column_F2
    """
    pred_tables, pred_columns = parse_prediction(predicted_text)

    pred_tables_norm = _normalize_tables(pred_tables)
    gold_tables_norm = _normalize_tables(set(gold_tables))

    pred_cols_norm = _normalize_columns(pred_columns)
    gold_cols_norm = _normalize_columns(set(gold_columns))

    table_f2 = _compute_f_beta(pred_tables_norm, gold_tables_norm)
    column_f2 = _compute_f_beta(pred_cols_norm, gold_cols_norm)

    content_score = 0.4 * table_f2 + 0.6 * column_f2
    format_score = _compute_format_score(predicted_text)
    total = 0.9 * content_score + 0.1 * format_score

    return total


# ---------- GRPOTrainer reward interface ----------

def reward_fn(completions, gold_tables, gold_columns, **kwargs):
    """Reward function for GRPOTrainer.

    GRPOTrainer calls this with:
      completions: list — model outputs (one per generation)
        - trl passes list[list[dict]]: completions[i][0]["content"]
        - some versions pass list[str] directly

    Returns: list[float] — one reward per completion
    """
    rewards = []
    for completion, g_tables, g_columns in zip(completions, gold_tables, gold_columns):
        # Handle both message-dict format and raw string format
        if isinstance(completion, list):
            text = completion[0]["content"]
        else:
            text = completion
        reward = compute_schema_linking_reward(text, g_tables, g_columns)
        rewards.append(reward)
    return rewards


# Quick test
test_pred = "Tables: invoice, customer\nColumns: invoice.total, customer.customer_id"
test_gold_t = ["invoice", "customer"]
test_gold_c = ["invoice.total", "customer.customer_id", "customer.first_name"]
test_reward = compute_schema_linking_reward(test_pred, test_gold_t, test_gold_c)
print(f"Test reward: {test_reward:.3f}")
print(f"  (Tables: perfect, Columns: 2/3 recall -> F2 penalises missing column)")
print("Reward function loaded.")

## 7. GRPOTrainer Setup

GRPO works by generating `NUM_GENERATIONS` candidate outputs for each prompt, scoring them with the reward function, and computing **group-relative advantages**: each candidate is scored relative to the group mean, not an absolute baseline. Candidates that score above the group mean get reinforced; those below get penalised.

Key differences from SFTTrainer:
- **No full target strings needed**: the reward function provides the training signal, but this setup still uses gold table/column labels
- **Multiple generations per prompt**: `num_generations=4` means 4 candidate outputs per question
- **Lower learning rate**: RL gradients are noisier than SFT; 5e-6 vs 2e-4
- **Shorter completions**: schema linking outputs are ~50-100 tokens, vs ~512 for SQL

In [ ]:
import os
os.environ["NPROC"] = "1"

from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,

    # GRPO-specific
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_COMPLETION_LENGTH,

    # Training
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,

    # Precision
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,

    # Misc
    optim="adamw_8bit",        # Recommended by Unsloth for Qwen3.5
    seed=3407,
    report_to="wandb" if USE_WANDB else "none",
    max_prompt_length=MAX_SEQ_LENGTH - MAX_COMPLETION_LENGTH,
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=reward_fn,
)

# --- Fix 1: Loss shape mismatch ---
# Unsloth's GRPO loss returns shape [1] but transformers 5.x _tr_loss
# accumulator is a scalar []. The in-place += fails on shape mismatch.
# Squeeze the loss to a scalar before it reaches the accumulator.
_orig_training_step = trainer.training_step

def _training_step_squeeze(model, inputs, num_items_in_batch=None):
    loss = _orig_training_step(model, inputs, num_items_in_batch)
    return loss.squeeze()

trainer.training_step = _training_step_squeeze

# --- Fix 2: coef_1 / completion_mask shape mismatch ---
# Unsloth's grpo_accumulated_loss internally handles left-padding by expanding
# completion tensors to (batch, logits_to_keep + max_left_pad). It returns
# coef_1 in this padded shape. But compute_loss uses the original completion_mask
# (shape logits_to_keep) for clip-ratio metrics, causing a broadcast error.
# Fix: truncate coef_1 to logits_to_keep before returning. The left-pad portion
# is already zero-masked internally, so truncating is safe.
_compute_loss_globals = trainer.compute_loss.__func__.__globals__
_orig_grpo_loss = _compute_loss_globals['grpo_accumulated_loss']

def _patched_grpo_accumulated_loss(*args, **kwargs):
    logits_to_keep = kwargs.get('logits_to_keep')
    result = _orig_grpo_loss(*args, **kwargs)
    if logits_to_keep is not None:
        result = list(result)
        coef_1 = result[-1]
        if hasattr(coef_1, 'shape') and coef_1.ndim >= 2 and coef_1.shape[1] > logits_to_keep:
            result[-1] = coef_1[:, -logits_to_keep:]
        result = tuple(result)
    return result

_compute_loss_globals['grpo_accumulated_loss'] = _patched_grpo_accumulated_loss

total_steps = (
    len(train_ds)
    // (BATCH_SIZE * GRAD_ACCUM_STEPS)
    * NUM_EPOCHS
)
print(f"Epochs: {NUM_EPOCHS}")
print(f"Effective batch: {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS} prompts")
print(f"Generations per prompt: {NUM_GENERATIONS}")
print(f"Training examples: {len(train_ds)}")
print(f"Estimated steps: ~{total_steps}")
print(f"Logging to: {'wandb' if USE_WANDB else 'console only'}")
print("Patches applied: loss squeeze + coef_1 truncation")

## 8. Train

Watch the reward increase during training. Unlike SFT loss (which decreases), GRPO reward should increase as the model learns to predict correct tables and columns.

Typical progression in small exploratory runs:
- **Early steps**: reward is low and noisy while the model learns the format
- **After format learning**: reward rises as table and column predictions improve
- **Convergence**: reward plateaus and step-to-step variance remains normal for RL

In [ ]:
import time

start_time = time.time()
train_result = trainer.train()
train_elapsed = time.time() - start_time

print(f"\nTraining complete in {train_elapsed/60:.1f} minutes")

## 9. Save LoRA Adapters

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapters saved to: {OUTPUT_DIR}")

# Optionally save merged model (base + adapters combined)
# model.save_pretrained_merged(
#     "output_grpo_schema_linking_merged",
#     tokenizer,
#     save_method="merged_16bit",
# )

## 10. Evaluate on Validation Set

Run the trained model on held-out validation examples and compute average reward, table F2, and column F2.

In [ ]:
FastVisionModel.for_inference(model)

n = min(EVAL_EXAMPLES, len(val_ds))
results = []

print(f"Evaluating trained model on {n} validation examples...\n")

for i in range(n):
    ex = val_ds[i]

    # Build prompt
    user_content = (
        f"<SCHEMA>\n{ex['ddl_context']}\n</SCHEMA>\n\n"
        f"<GLOSSARY>\n{ex['glossary_context']}\n</GLOSSARY>\n\n"
        f"Question: {ex['question']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(generated, skip_special_tokens=True).strip()

    # Score
    pred_tables, pred_columns = parse_prediction(output_text)
    pred_t_norm = _normalize_tables(pred_tables)
    gold_t_norm = _normalize_tables(set(ex["gold_tables"]))
    pred_c_norm = _normalize_columns(pred_columns)
    gold_c_norm = _normalize_columns(set(ex["gold_columns"]))

    table_f2 = _compute_f_beta(pred_t_norm, gold_t_norm)
    column_f2 = _compute_f_beta(pred_c_norm, gold_c_norm)
    reward = compute_schema_linking_reward(output_text, ex["gold_tables"], ex["gold_columns"])

    results.append({
        "reward": reward, "table_f2": table_f2, "column_f2": column_f2,
        "output": output_text, "question": ex["question"],
        "pred_tables": sorted(pred_t_norm), "gold_tables": sorted(gold_t_norm),
        "pred_columns": sorted(pred_c_norm), "gold_columns": sorted(gold_c_norm),
    })

    if i < 5:  # Show first 5 examples
        print(f"--- Example {i+1} ---")
        print(f"  Q: {ex['question'][:80]}...")
        print(f"  Output: {output_text[:120]}...")
        print(f"  Tables:  pred={sorted(pred_t_norm)}  gold={sorted(gold_t_norm)}  F2={table_f2:.2f}")
        print(f"  Columns: pred={sorted(pred_c_norm)[:3]}...  gold={sorted(gold_c_norm)[:3]}...  F2={column_f2:.2f}")
        print(f"  Reward: {reward:.3f}")
        print()

# Summary
avg_reward = sum(r["reward"] for r in results) / n
avg_table_f2 = sum(r["table_f2"] for r in results) / n
avg_column_f2 = sum(r["column_f2"] for r in results) / n

print(f"{'='*50}")
print(f"  GRPO-trained model ({n} val examples)")
print(f"  Average reward:    {avg_reward:.3f}")
print(f"  Average table F2:  {avg_table_f2:.3f}")
print(f"  Average column F2: {avg_column_f2:.3f}")
print(f"{'='*50}")

# Log eval metrics to wandb
if USE_WANDB and wandb.run is not None:
    wandb.log({
        "eval/avg_reward": avg_reward,
        "eval/avg_table_f2": avg_table_f2,
        "eval/avg_column_f2": avg_column_f2,
        "eval/n_examples": n,
    })
    print("Eval metrics logged to wandb.")

## 11. Compare: Base Model vs GRPO-Trained

Load a fresh base model (no LoRA) to measure the improvement from GRPO training.

In [ ]:
# Load fresh base model for comparison
print("Loading base model (without LoRA)...")
base_model, base_processor = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-2B",
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
)
base_tokenizer = base_processor.tokenizer
FastVisionModel.for_inference(base_model)

n = min(EVAL_EXAMPLES, len(val_ds))
print(f"Comparing on {n} validation examples...\n")

base_results = []

for i in range(n):
    ex = val_ds[i]
    user_content = (
        f"<SCHEMA>\n{ex['ddl_context']}\n</SCHEMA>\n\n"
        f"<GLOSSARY>\n{ex['glossary_context']}\n</GLOSSARY>\n\n"
        f"Question: {ex['question']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    prompt = base_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = base_tokenizer(prompt, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=MAX_COMPLETION_LENGTH,
            do_sample=False,
            pad_token_id=base_tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    output_text = base_tokenizer.decode(generated, skip_special_tokens=True).strip()

    reward = compute_schema_linking_reward(output_text, ex["gold_tables"], ex["gold_columns"])
    pred_tables, pred_columns = parse_prediction(output_text)
    table_f2 = _compute_f_beta(
        _normalize_tables(pred_tables), _normalize_tables(set(ex["gold_tables"]))
    )
    column_f2 = _compute_f_beta(
        _normalize_columns(pred_columns), _normalize_columns(set(ex["gold_columns"]))
    )

    base_results.append({"reward": reward, "table_f2": table_f2, "column_f2": column_f2})

# Free base model
del base_model, base_processor, base_tokenizer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Compare
b_reward = sum(r["reward"] for r in base_results) / n
b_table = sum(r["table_f2"] for r in base_results) / n
b_column = sum(r["column_f2"] for r in base_results) / n
f_reward = sum(r["reward"] for r in results) / n
f_table = sum(r["table_f2"] for r in results) / n
f_column = sum(r["column_f2"] for r in results) / n

print(f"{'='*60}")
print(f"  {'Metric':<20} {'Base':<15} {'GRPO-trained':<15}")
print(f"  {'-'*20} {'-'*15} {'-'*15}")
print(f"  {'Avg reward':<20} {b_reward:.3f}{'':>12} {f_reward:.3f}")
print(f"  {'Avg table F2':<20} {b_table:.3f}{'':>12} {f_table:.3f}")
print(f"  {'Avg column F2':<20} {b_column:.3f}{'':>12} {f_column:.3f}")
print(f"{'='*60}")

# Log comparison to wandb and finish the run
if USE_WANDB and wandb.run is not None:
    wandb.log({
        "compare/base_reward": b_reward,
        "compare/base_table_f2": b_table,
        "compare/base_column_f2": b_column,
        "compare/grpo_reward": f_reward,
        "compare/grpo_table_f2": f_table,
        "compare/grpo_column_f2": f_column,
        "compare/reward_improvement": f_reward - b_reward,
    })
    wandb.finish()
    print("wandb run finished.")

## 11b. Compare with GPT-4.1-mini

How does a 2B GRPO-trained model compare against a frontier cloud model on the same task?
GPT-4.1-mini serves as a strong baseline — it has far more parameters and was trained on
vastly more data, but costs money per call and adds latency. Set `OPENAI_API_KEY`; optionally
override `OPENAI_API_BASE` if you use a different OpenAI-compatible endpoint.

In [ ]:
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
    base_url=os.environ.get("OPENAI_API_BASE", "https://api.openai.com/v1"),
)
GPT_MODEL = "gpt-4.1-mini"

n = min(EVAL_EXAMPLES, len(val_ds))
print(f"Evaluating {GPT_MODEL} on {n} validation examples...\n")

gpt_results = []

for i in range(n):
    ex = val_ds[i]
    user_content = (
        f"<SCHEMA>\n{ex['ddl_context']}\n</SCHEMA>\n\n"
        f"<GLOSSARY>\n{ex['glossary_context']}\n</GLOSSARY>\n\n"
        f"Question: {ex['question']}"
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]

    try:
        response = client.chat.completions.create(
            model=GPT_MODEL,
            messages=messages,
            temperature=0,
            max_tokens=512,
        )
        output_text = response.choices[0].message.content.strip()
    except Exception as e:
        print(f"  Example {i+1} failed: {e}")
        output_text = ""

    # Score with the same reward function
    pred_tables, pred_columns = parse_prediction(output_text)
    pred_t_norm = _normalize_tables(pred_tables)
    gold_t_norm = _normalize_tables(set(ex["gold_tables"]))
    pred_c_norm = _normalize_columns(pred_columns)
    gold_c_norm = _normalize_columns(set(ex["gold_columns"]))

    table_f2 = _compute_f_beta(pred_t_norm, gold_t_norm)
    column_f2 = _compute_f_beta(pred_c_norm, gold_c_norm)
    reward = compute_schema_linking_reward(output_text, ex["gold_tables"], ex["gold_columns"])

    gpt_results.append({
        "reward": reward, "table_f2": table_f2, "column_f2": column_f2,
        "output": output_text, "question": ex["question"],
        "pred_tables": sorted(pred_t_norm), "gold_tables": sorted(gold_t_norm),
        "pred_columns": sorted(pred_c_norm), "gold_columns": sorted(gold_c_norm),
    })

    if i < 3:
        print(f"--- Example {i+1} ---")
        print(f"  Q: {ex['question'][:80]}...")
        print(f"  Output: {output_text[:120]}...")
        print(f"  Tables:  F2={table_f2:.2f}  Columns: F2={column_f2:.2f}  Reward: {reward:.3f}")
        print()

# Summary
g_reward = sum(r["reward"] for r in gpt_results) / n
g_table = sum(r["table_f2"] for r in gpt_results) / n
g_column = sum(r["column_f2"] for r in gpt_results) / n

# Recompute base and GRPO from earlier cells
b_reward = sum(r["reward"] for r in base_results) / n
b_table = sum(r["table_f2"] for r in base_results) / n
b_column = sum(r["column_f2"] for r in base_results) / n
f_reward = sum(r["reward"] for r in results) / n
f_table = sum(r["table_f2"] for r in results) / n
f_column = sum(r["column_f2"] for r in results) / n

print(f"\n{'='*76}")
print(f"  {'Metric':<20} {'Base (2B)':<17} {'GRPO-trained (2B)':<20} {GPT_MODEL:<15}")
print(f"  {'-'*20} {'-'*17} {'-'*20} {'-'*15}")
print(f"  {'Avg reward':<20} {b_reward:<17.3f} {f_reward:<20.3f} {g_reward:.3f}")
print(f"  {'Avg table F2':<20} {b_table:<17.3f} {f_table:<20.3f} {g_table:.3f}")
print(f"  {'Avg column F2':<20} {b_column:<17.3f} {f_column:<20.3f} {g_column:.3f}")
print(f"{'='*76}")

# Highlight the story
if f_reward > g_reward:
    diff = f_reward - g_reward
    print(f"\n  GRPO-trained 2B model beats {GPT_MODEL} by {diff:+.3f} reward!")
elif f_reward > g_reward - 0.05:
    print(f"\n  GRPO-trained 2B model is competitive with {GPT_MODEL} (within 0.05)!")
else:
    diff = g_reward - f_reward
    print(f"\n  {GPT_MODEL} leads by {diff:+.3f} — but at ~100x the cost per query.")

## 12. Summary

**What you did:**
- Trained `Qwen3.5-2B` to predict schema elements (tables + columns) using GRPO
- Used a reward function (F2 score) instead of labeled examples — the model learned *what works* rather than imitating
- GRPOTrainer generated multiple candidates per prompt, scored them, and reinforced the best

**Track 1 (SFT) vs Track 2 (GRPO):**
- SFT teaches *format* by imitating examples. GRPO optimises *content* by rewarding correct predictions.
- SFT needs labeled `(input, output)` pairs. GRPO needs only inputs + a reward function.
- SFT quality is bounded by training data. GRPO can discover strategies not in the training data.

**This notebook vs Chapter 4C.3:**
- This notebook uses Unsloth + GRPOTrainer on a **single GPU**. Good for exploration and smaller models.
- Chapter 4C.3 uses VERL + Agent Lightning for **multi-GPU training** with vLLM rollouts. Needed for production scale and multi-turn agent training.
- Unsloth handles Qwen3.5-2B's hybrid DeltaNet architecture. VERL requires standard transformer architectures.

**Next step — Chapter 4C**: Scale GRPO training with VERL + Agent Lightning, add multi-turn tool-calling agents, and train on the full SQL generation task.